In [ ]:
# Run this cell to import pyspark and to define start_spark() and stop_spark()

import findspark

findspark.init()

import getpass
import pyspark
import random
import re

from IPython.display import display, HTML
from pyspark import SparkContext
from pyspark.sql import SparkSession


# Constants used to interact with Azure Blob Storage using the hdfs command or Spark

global username

username = re.sub('@.*', '', getpass.getuser())


# Functions used below

def dict_to_html(d):
    """Convert a Python dictionary into a two column table for display.
    """

    html = []

    html.append(f'<table width="100%" style="width:100%; font-family: monospace;">')
    for k, v in d.items():
        html.append(f'<tr><td style="text-align:left;">{k}</td><td>{v}</td></tr>')
    html.append(f'</table>')

    return ''.join(html)


def show_as_html(df, n=20):
    """Leverage existing pandas jupyter integration to show a spark dataframe as html.
    
    Args:
        n (int): number of rows to show (default: 20)
    """

    display(df.limit(n).toPandas())

    
def display_spark():
    """Display the status of the active Spark session if one is currently running.
    """
    
    if 'spark' in globals() and 'sc' in globals():

        name = sc.getConf().get("spark.app.name")

        html = [
            f'<p><b>Spark</b></p>',
            f'<p>The spark session is <b><span style="color:green">active</span></b>, look for <code>{name}</code> under the running applications section in the Spark UI.</p>',
            f'<ul>',
            f'<li><a href="http://localhost:{sc.uiWebUrl.split(":")[-1]}" target="_blank">Spark Application UI</a></li>',
            f'</ul>',
            f'<p><b>Config</b></p>',
            dict_to_html({k: v for k, v in sc.getConf().getAll() if not re.search(r"(secret|password|token|credential|sas|account\.key)", k, re.I)}),
            f'<p><b>Notes</b></p>',
            f'<ul>',
            f'<li>The spark session <code>spark</code> and spark context <code>sc</code> global variables have been defined by <code>start_spark()</code>.</li>',
            f'<li>Please run <code>stop_spark()</code> before closing the notebook or restarting the kernel or kill <code>{name}</code> by hand using the link in the Spark UI.</li>',
            f'</ul>',
        ]
        display(HTML(''.join(html)))
        
    else:
        
        html = [
            f'<p><b>Spark</b></p>',
            f'<p>The spark session is <b><span style="color:red">stopped</span></b>, confirm that <code>{username} (notebook)</code> is under the completed applications section in the Spark UI.</p>',
            f'<ul>',
            f'<li><a href="http://mathmadslinux2p.canterbury.ac.nz:8080/" target="_blank">Spark UI</a></li>',
            f'</ul>',
        ]
        display(HTML(''.join(html)))


# Functions to start and stop spark

def start_spark(executor_instances=2, executor_cores=1, worker_memory=1, master_memory=1):
    """Start a new Spark session and define globals for SparkSession (spark) and SparkContext (sc).
    
    Args:
        executor_instances (int): number of executors (default: 2)
        executor_cores (int): number of cores per executor (default: 1)
        worker_memory (float): worker memory (default: 1)
        master_memory (float): master memory (default: 1)
    """

    global spark
    global sc

    cores = executor_instances * executor_cores
    partitions = cores * 4
    port = 4000 + random.randint(1, 999)

    spark = (
        SparkSession.builder
        .config("spark.driver.extraJavaOptions", f"-Dderby.system.home=/tmp/{username}/spark/")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.executor.instances", str(executor_instances))
        .config("spark.executor.cores", str(executor_cores))
        .config("spark.cores.max", str(cores))
        .config("spark.driver.memory", f'{master_memory}g')
        .config("spark.executor.memory", f'{worker_memory}g')
        .config("spark.driver.maxResultSize", "0")
        .config("spark.sql.shuffle.partitions", str(partitions))
        .config("spark.kubernetes.container.image", "madsregistry001.azurecr.io/hadoop-spark:v3.3.5-openjdk-8")
        .config("spark.kubernetes.container.image.pullPolicy", "IfNotPresent")
        .config("spark.kubernetes.memoryOverheadFactor", "0.3")
        .config("spark.memory.fraction", "0.1")
        .config("spark.app.name", f"{username} (notebook)")
        .getOrCreate()
    )
    sc = SparkContext.getOrCreate()
    
    display_spark()

    
def stop_spark():
    """Stop the active Spark session and delete globals for SparkSession (spark) and SparkContext (sc).
    """

    global spark
    global sc

    if 'spark' in globals() and 'sc' in globals():

        spark.stop()

        del spark
        del sc

    display_spark()


# Make css changes to improve spark output readability

html = [
    '<style>',
    'pre { white-space: pre !important; }',
    'table.dataframe td { white-space: nowrap !important; }',
    'table.dataframe thead th:first-child, table.dataframe tbody th { display: none; }',
    '</style>',
]
display(HTML(''.join(html)))

In [ ]:
# Run this cell to start a spark session in this notebook

start_spark(executor_instances=4, executor_cores=4, worker_memory=4, master_memory=4)

In [ ]:
# We need to import pyplot from matplotlib in order to visualize our data locally 

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

from pyspark.sql import Row, DataFrame, Window, functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import rand, row_number, when, col, array_intersect, size, countDistinct, count, mean, stddev, min, max, floor, collect_list, collect_set, collect_list, explode, slice, lit, expr

from pyspark.ml.functions import vector_to_array
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.sql.types import NumericType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.feature import StringIndexer
from pyspark.ml.functions import vector_to_array
from pyspark.mllib.evaluation import RankingMetrics

import numpy as np

In [ ]:
# Other imports to be used locally

import datetime
import numpy as np

np.set_printoptions(edgeitems=5, threshold=100, precision=4)

In [ ]:
# Determine ideal number of partitions

conf = sc.getConf()

N = int(conf.get("spark.executor.instances"))
M = int(conf.get("spark.executor.cores"))
partitions = 4 * N * M

print(f'ideal # partitions = {partitions}')

# Q1

## b)

In [ ]:
# Define the schema of triplets file
triplets_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("song_id", StringType(), True),
    StructField("play_count", IntegerType(), True)
])
# Load a TSV file
triplets = (
    spark.read.format("csv")
    .option("header", "false")
    .option("inferSchema", "true")
    .option("sep", "\t")
    .schema(triplets_schema)
    .load("wasbs://campus-data@madsstorage002.blob.core.windows.net/msd/tasteprofile/triplets.tsv")
    .repartition(64).cache()
)
triplets.printSchema()
show_as_html(triplets)

In [ ]:
# Calculate unique songs and users
unique_songs = triplets.select("song_id").distinct().count()
unique_users = triplets.select("user_id").distinct().count()

# Print results
print("Unique songs:", unique_songs)
print("Unique users:", unique_users)

In [ ]:
from pyspark.sql.functions import sum as sum_, desc, countDistinct

# 1. Find, for each user, their total play count
user_total_plays = (
    triplets
    .groupBy("user_id")
    .agg(sum_("play_count").alias("total_plays"))
)

# 2. Identify the user with the maximum total plays
most_active = (
    user_total_plays
    .orderBy(desc("total_plays"))
    .first()
)
most_active_user_id = most_active["user_id"]

# 3. For that user, count the number of distinct songs they’ve ever played
unique_songs_by_most_active = (
    triplets
    .filter(triplets.user_id == most_active_user_id)
    .select("song_id")
    .distinct()
    .count()
)

# 4. Calculate the percentage
percentage_of_total = (unique_songs_by_most_active / unique_songs) * 100

print("Most active user by total plays:", most_active_user_id)
print("Total plays by that user:", most_active["total_plays"])
print("Unique songs played by that user:", unique_songs_by_most_active)
print(f"Percentage of total songs: {percentage_of_total:.2f}%")

## c)

In [ ]:
# Compute total play count per song
song_total_plays = triplets.groupBy("song_id") \
    .agg(sum_("play_count").alias("total_play_count"))

# Compute total play count per user
user_total_plays = triplets.groupBy("user_id") \
    .agg(sum_("play_count").alias("total_play_count"))

# Show summary statistics for total play counts per song
song_total_plays.select("total_play_count") \
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max") \
    .show()

# Show summary statistics for total play counts per user
user_total_plays.select("total_play_count") \
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max") \
    .show()


In [ ]:
# 1. Compute number of unique listeners per song
song_distinct_user = (
    triplets
    .groupBy("song_id")
    .agg(countDistinct("user_id").alias("user_count"))
)

# 2. Compute number of unique songs per user
user_distinct_song = (
    triplets
    .groupBy("user_id")
    .agg(countDistinct("song_id").alias("song_count"))
)

# 3. Show descriptive stats for song popularity
#    summary() supports count, mean, stddev, min, quartiles, max
song_distinct_user.select("user_count") \
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max") \
    .show()

# 4. Show descriptive stats for user activity
user_distinct_song.select("song_count") \
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max") \
    .show()


## d)

In [ ]:
# Collect total plays per song for plotting
song_counts = song_total_plays.select("total_play_count") \
    .rdd.flatMap(lambda row: row).collect()

# Plot histogram for song total plays
plt.figure()
plt.hist(song_counts, bins=50)
plt.title("Distribution of Total Play Counts per Song")
plt.xlabel("Total Play Count")
plt.ylabel("Number of Songs")
plt.savefig("song_total_plays_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Collect total plays per user for plotting
user_counts = user_total_plays.select("total_play_count") \
    .rdd.flatMap(lambda row: row).collect()

# Plot histogram for user total plays
plt.figure()
plt.hist(user_counts, bins=50)
plt.title("Distribution of Total Play Counts per User")
plt.xlabel("Total Play Count")
plt.ylabel("Number of Users")
plt.savefig("user_total_plays_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Collect counts for plotting
pop_counts = song_distinct_user.select("user_count").rdd.flatMap(lambda x: x).collect()
act_counts = user_distinct_song.select("song_count").rdd.flatMap(lambda x: x).collect()

# Plot distribution of song popularity
plt.figure()
plt.hist(pop_counts, bins=50)
plt.title("Distribution of distinct users of Songs")
plt.xlabel("Number of Unique Listeners")
plt.ylabel("Number of Songs")
plt.savefig("distinct_users_of_each_song_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Plot distribution of user activity
plt.figure()
plt.hist(act_counts, bins=50)
plt.title("Distribution of distinct songs of Users")
plt.xlabel("Number of Unique Songs Listened")
plt.ylabel("Number of Users")
plt.savefig("distinct_songs_of_each_user_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## Q2

### a)

In [ ]:
from pyspark.sql.functions import sum as sum_, countDistinct

# Set thresholds
N = 20  # minimum total play count per song
M = 20  # minimum distinct songs per user

# Count original totals
orig_song_count = triplets.select("song_id").distinct().count()
orig_user_count = triplets.select("user_id").distinct().count()

# 1. Compute distinct users per song and show its distribution
song_totals = (
    triplets
    .groupBy("song_id")
    # .agg(sum_("play_count").alias("total_plays"))
    .agg(countDistinct("user_id").alias("distinct_users"))
)

# 2. Filter out songs with fewer than N total plays
valid_songs = (
    song_totals
    .filter(song_totals.distinct_users >= N)
    .select("song_id")
)
filtered1 = triplets.join(valid_songs, on="song_id")

# 3. Compute distinct songs per user and show its distribution
user_totals = (
    filtered1
    .groupBy("user_id")
    .agg(countDistinct("song_id").alias("distinct_songs"))
)

# 4. Filter out users with fewer than M distinct songs
valid_users = (
    user_totals
    .filter(user_totals.distinct_songs >= M)
    .select("user_id")
)
filtered_final = filtered1.join(valid_users, on="user_id")

# 5. Compute remaining and excluded counts
remaining_song_count = filtered_final.select("song_id").distinct().count()
remaining_user_count = filtered_final.select("user_id").distinct().count()
excluded_songs = orig_song_count - remaining_song_count
excluded_users = orig_user_count - remaining_user_count

# 6. Show summary table
summary = spark.createDataFrame([
    ("songs", orig_song_count, remaining_song_count, excluded_songs),
    ("users", orig_user_count, remaining_user_count, excluded_users)
], ["entity", "original", "remaining", "excluded"])
summary.show()


In [ ]:
filtered_final.cache()
filtered_final.count()

### b)

In [ ]:
# 1. Fit a StringIndexer model for users
user_indexer = StringIndexer() \
    .setInputCol("user_id") \
    .setOutputCol("user_idx") \
    .fit(filtered_final)

# 2. Apply it to add a user_idx column
indexed = user_indexer.transform(filtered_final)

# 3. Fit a second StringIndexer model for songs on the already-indexed data
song_indexer = StringIndexer() \
    .setInputCol("song_id") \
    .setOutputCol("song_idx") \
    .fit(indexed)

# 4. Apply it to add a song_idx column
indexed = song_indexer.transform(indexed)

# 5. Cast the generated indices (which are doubles) to integers
indexed = indexed \
    .withColumn("user_idx", col("user_idx").cast("int")) \
    .withColumn("song_idx", col("song_idx").cast("int"))

# 6.  drop the original string columns if you only want numeric IDs
final_df = indexed.drop("user_id", "song_id")

# Inspect schema
final_df.printSchema()
show_as_html(final_df)

### c)

In [ ]:
# Step 1: assign a random number to each row
df_with_rand = final_df.withColumn("rand", F.rand())

# Step 2: for each user, rank their plays by the random number
user_win = Window.partitionBy("user_idx").orderBy("rand")
df_ranked = df_with_rand.withColumn("rank", F.row_number().over(user_win))

# Step 3: calculate total number of plays per user
count_win = Window.partitionBy("user_idx")
df_counted = df_ranked.withColumn("user_play_count", F.count("*").over(count_win))

# Step 4: determine the split point (80% of each user’s plays, at least 1)
df_split = df_counted.withColumn(
    "split_point",
    F.greatest((F.col("user_play_count") * 0.8).cast("int"), F.lit(1))
)

# Step 5: build train and test
train_df = df_split.filter(F.col("rank") <= F.col("split_point")) \
                   .drop("rand", "rank", "user_play_count", "split_point") \
                   .repartition(200, "user_idx", "song_idx") \
                   .cache()
test_df  = df_split.filter(F.col("rank")  > F.col("split_point")) \
                   .drop("rand", "rank", "user_play_count", "split_point") \
                   .repartition("user_idx") \
                   .cache()

# View split sizes
print("Train count:", train_df.count())
print("Test  count:", test_df.count())


### d)

In [ ]:
# configure ALS for implicit feedback
als = ALS(
    userCol="user_idx",
    itemCol="song_idx",
    ratingCol="play_count",
    implicitPrefs=True,
    rank=20,                 
    regParam=0.05,           
    alpha=20.0,             
    maxIter=15,    
    seed=25,
    coldStartStrategy="drop"
)

# fit the model on the training data
als_model = als.fit(train_df)

# inspect a few user factors and item factors
print("User factors schema:", als_model.userFactors.schema)
print("Item factors schema:", als_model.itemFactors.schema)

# optionally cache the factors if you’ll use them a lot
als_model.userFactors.cache()
als_model.itemFactors.cache()

In [ ]:
from pyspark.sql.functions import collect_set, collect_list, col, explode

# 1. Pick three users at random from the test set
sample_users = test_df.select("user_idx").distinct().limit(3)

# 2. Ask the model for top-5 recommendations per user
recs = als_model.recommendForUserSubset(sample_users, 10)

# 3. Explode into (user, song, score) rows
recs_exploded = recs.select(
    col("user_idx"),
    explode(col("recommendations")).alias("rec")
).select(
    "user_idx",
    col("rec.song_idx").alias("recommended_song"),
    col("rec.rating").alias("score")
)

# 4. Turn those back into a list per user
recs_list = recs_exploded.groupBy("user_idx") \
    .agg(collect_list("recommended_song").alias("recommended_songs"))

# 5. Gather actual songs each user played in the test set
actual_list = test_df.groupBy("user_idx") \
    .agg(collect_set("song_idx").alias("actual_songs"))

# 6. Join them for side-by-side comparison
comparison = recs_list.join(actual_list, on="user_idx")\
    .repartition("user_idx") \
    .cache()

# 7. View
comparison.show(truncate=False)


In [ ]:
from pyspark.sql.functions import expr, size, slice, col, lit

# 9. Compute the list of correctly recommended songs (intersection), count of correct recommendations, and Precision@10,
#    and extract the top 10 actual songs
evaluation_metrics = comparison.withColumn(
    "correct_songs",
    expr("array_intersect(recommended_songs, actual_songs)")
).withColumn(
    "correct_count",
    size(col("correct_songs"))
).withColumn(
    "precision_at_10",
    col("correct_count") / size(col("recommended_songs"))
).withColumn(
    "actual_top_10",
    slice(col("actual_songs"), 1, 10)
).select(
    "user_idx",
    "recommended_songs",
    "actual_top_10",
    "precision_at_10",
    "correct_count"
)

# 10. Display the results
evaluation_metrics.show(truncate=False)


### f)

In [ ]:
from pyspark.sql.functions import expr, col
from pyspark.sql.functions import transform
from pyspark.ml.evaluation import RankingEvaluator

# 1. Convert integer arrays to double arrays
comparison_double = comparison \
    .withColumn(
        "predictions_double",
        expr("transform(recommended_songs, x -> cast(x as double))")
    ) \
    .withColumn(
        "labels_double",
        expr("transform(actual_songs, x -> cast(x as double))")
    ) \
    .repartition("user_idx") \
    .cache()

# 2. Trigger cache by counting
comparison_double.count()

# 3. Create RankingEvaluators for each metric (all at K=10)
precision_evaluator = RankingEvaluator(
    predictionCol="predictions_double",  # use the casted column
    labelCol="labels_double",
    metricName="precisionAtK",
    k=10
)
ndcg_evaluator = RankingEvaluator(
    predictionCol="predictions_double",
    labelCol="labels_double",
    metricName="ndcgAtK",
    k=10
)
map_evaluator = RankingEvaluator(
    predictionCol="predictions_double",
    labelCol="labels_double",
    metricName="meanAveragePrecision",
    k=10
)

# 4. Evaluate and print results
precision_at_10 = precision_evaluator.evaluate(comparison_double)
ndcg_at_10      = ndcg_evaluator.evaluate(comparison_double)
map_at_10       = map_evaluator.evaluate(comparison_double)

print(f"Precision@10: {precision_at_10:.4f}")
print(f"NDCG@10:      {ndcg_at_10:.4f}")
print(f"MAP@10:       {map_at_10:.4f}")


In [ ]:
stop_spark()